# Calling the OpenAI API

Working up from one plain call to a conversation that remembers what was said.

## Load the key

The key lives in `.env` at the repo root, never in the notebook - this repo is public.
Printing the boolean just confirms it loaded, without showing anything.

In [ ]:
import os
from dotenv import load_dotenv

# .env lives at the repo root, two levels up from this notebook
load_dotenv("../../.env")

key = os.environ.get("OPENAI_API_KEY")
# never print the key itself - this repo is public
print("key loaded:", bool(key), "| length:", len(key) if key else 0)

## First call

`responses.create` is the current API. `output_text` is the shortcut that skips the nesting and
hands you the answer as a plain string.

In [ ]:
from openai import OpenAI

openai_client = OpenAI()

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Tell me a joke"
)

print(response.output_text)

The full response, to see what else comes back besides the text - id, model, token counts.

In [ ]:
print(response.model_dump_json(indent=2))

## Passing messages instead of a string

Same result, but a list of `role`/`content` dicts is what you need the moment there's more than
one turn to send.

In [ ]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

In [ ]:
print(response.model_dump_json(indent=2))

`usage` is the token count, which is what you actually pay for. Worth a look.

In [ ]:
response.usage

In [ ]:
print(response.output_text)

## Streaming

Prints the answer as it arrives instead of waiting for the whole thing. Better when the reply
is long and someone is sitting there watching it.

In [ ]:
stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')

In [ ]:
response.usage

## System instructions

The `system` message is standing orders for the model, `user` is the actual request.
Here the standing order is to add emojis, and it applies to whatever we ask next.

In [ ]:
instructions = "make sure to add emojis"
user_prompt = "tell me a bad time story about a unicorn"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": user_prompt},
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

In [ ]:
response.usage

In [ ]:
print(response.output_text)

In [ ]:
instructions = "make sure to add emojis"
user_prompt = "my name is Manoj"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": user_prompt},
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

print(response.output_text)

## The model doesn't remember

Fresh call, so it has never seen the message where we gave our name. Nothing is stored between
calls - each one starts from nothing.

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input="What's my name?"
)

print(response.output_text)

## Giving it memory by hand

Append the model's own reply onto `messages`, then ask again. The "memory" is nothing clever:
we just resend the whole conversation every time.

In [ ]:
messages.extend(response.output)

In [ ]:
messages

In [ ]:
messages.append(
    {"role": "user", "content": "what's my name?"}
)

In [ ]:
messages

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

print(response.output_text)